<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# Build a Multi-Agent Chatbot with AG2 (AutoGen) for Healthcare


Estimated time needed: **30** minutes


AutoMed is not just an ordinary chatbot—it’s a multi-agent AI system powered by AG2 (AutoGen), designed to simulate expert medical consultation through intelligent collaboration. Instead of relying on a single AI agent, AutoMed orchestrates multiple specialized agents, each dedicated to a specific task, ensuring comprehensive, accurate, and real-time medical guidance. By leveraging AutoGen’s multi-agent capabilities, AutoMed mimics the behavior of a real medical team, where different AI agents collaborate to analyze symptoms, suggest treatments, fetch real-time medical data, and provide follow-up care.

With its adaptive intelligence and multi-agent communication, AutoMed delivers human-like, context-aware conversations that go beyond basic symptom checkers. Unlike conventional AI chatbots that provide one-size-fits-all responses, AutoMed's specialized agents work together to deliver precise, tailored recommendations based on the user’s health history and real-time input. This results in a more interactive, intelligent, and reliable medical consultation experience.

<p style="color:red;"> Disclaimer: This guided project is designed to introduce learners to AG2 (AutoGen). The medical advice provided should not be considered a substitute for professional medical consultation, diagnosis, or treatment. Always seek guidance from a qualified healthcare professional.</p>


## __Table of Contents__

<ol>
    <li><a href="#Objectives">Objectives</a></li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Installing-Required-Libraries">Installing Required Libraries</a></li>
            <li><a href="#Importing-Required-Libraries">Importing Required Libraries</a></li>
        </ol>
    </li>
    <li><a href="#What-is-AutoGen?">What is AutoGen?</a></li>
    <li><a href="#Key-Features-of-AutoGen">Key Features of AutoGen</a></li>
    <li><a href="#Comparison:-AutoGen-vs-Traditional-AI-Agents">Comparison: AutoGen vs Traditional AI Agents</a></li>
    <li><a href="#How-AutoMed-Works:-Multi-Agent-AI-in-Action">How AutoMed Works: Multi-Agent AI in Action</a></li>
    <li><a href="#Why-is-GPT-4o-Used?">Why is GPT-4o Used?</a></li>
    <li><a href="#What-is--ConversableAgent?">What is ConversableAgent?</a></li>
    <li><a href="#What-is-GroupChat?">What is GroupChat?</a></li>
    <li><a href="#Exercise:-Create-a-Mental-Health-Chatbot-Using-the-AutoGen-Library">Exercise: Create a Mental Health Chatbot Using the AutoGen Library</a></li>
</ol>

<ul>
    <li><a href="#Authors">Authors</a></li>
    <li><a href="#Other-Contributors">Contributors</a></li>
    <li><a href="#Change-Log">Change Log</a></li>
</ul>




## Objectives

After completing this lab you will be able to:

- Learn how AG2 (AutoGen) enables multi-agent AI systems for complex workflows.
- Explore how AG2 (AutoGen) integrates with LLMs like GPT-4 for dynamic AI-driven conversations.
- Implement agent-to-agent communication for intelligent medical decision-making.
- Develop multiple AI agents that interact and collaborate to handle different healthcare tasks.


----


## Setup


For this lab, we will use the following libraries from the project environment:

* [`AG2 (AutoGen)`](https://docs.ag2.ai/) for orchestrating multi-agent conversations
* [`python-dotenv`](https://pypi.org/project/python-dotenv/) for loading `OPENAI_API_KEY` and optional `OPENAI_MODEL` values from a `.env` file
* OpenAI models through AG2's `LLMConfig`

Dependencies are declared in the repository-level `requirements.in` file. The notebook does not install libraries at runtime.


### Required Libraries

This notebook expects the repository environment to be installed from `requirements.in`. Do not run package installation commands inside the notebook.


In [1]:
# Dependencies are managed in ../../requirements.in:
# ag2[openai], openai, python-dotenv
print("Using project requirements; no notebook-level package installation is needed.")


Using project requirements; no notebook-level package installation is needed.


### Importing Required Libraries
Import all required libraries:


In [2]:
import logging
import os
import warnings

from dotenv import load_dotenv

from autogen import ConversableAgent, LLMConfig
from autogen.agentchat import run_group_chat
from autogen.agentchat.group import AgentTarget, ContextVariables, TerminateTarget
from autogen.agentchat.group.patterns import DefaultPattern

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("autogen.oai.client").setLevel(logging.ERROR)

load_dotenv()

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-5-nano")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in your environment or .env file before running this notebook.")

llm_config = LLMConfig(
    {
        "api_type": "openai",
        "model": MODEL_NAME,
        "api_key": OPENAI_API_KEY,
    }
)

print(f"AG2 configured for OpenAI model: {MODEL_NAME}")


AG2 configured for OpenAI model: gpt-5-nano


In [3]:
# Warning suppression is configured in the import cell above.


# What is AutoGen?

AutoGen is an open-source framework developed by Microsoft that enables developers to orchestrate and optimize AI workflows using multiple AI agents. These agents can collaborate, automate decision-making, and dynamically generate responses in complex problem-solving tasks.

Unlike traditional AI systems that work in isolation, AutoGen enables multiple AI agents (LLMs such as GPT) to interact, exchange information, and refine their outputs, making it more powerful and flexible for various applications.

# Key Features of AutoGen
### 1. Multi-Agent Collaboration

AutoGen allows multiple AI agents to communicate and solve tasks collaboratively. Each agent can have a specific role, such as a problem solver, verifier, or optimizer.

Example:

- One agent generates code, another reviews it, and another tests it.
- A research agent collects information, while another summarizes it.

### 2. Conversational and Task-Oriented AI

AutoGen supports LLM-driven conversations where AI agents engage in multi-turn dialogues to refine answers.

Example:

- A chatbot that consults different AI agents, such as one for legal advice and another for finance.
- A customer support AI that escalates unresolved queries to another AI agent.

### 3. Automated Workflow Generation

You can orchestrate workflows for AI-driven automation, such as AI-assisted programming, research, and document generation.

Example:

- Automating software debugging where AI identifies issues, suggests fixes, and verifies solutions.

### 4. Supports Human-AI Collaboration

AutoGen allows humans to intervene in AI-driven workflows by providing feedback or manually guiding agents when necessary.

Example:

- A research assistant AI drafts a report, but a human expert refines it.








# Comparison: AutoGen vs Traditional AI Agents

<table width="60%" align="left" >
        <thead>
            <tr>
                <th>Feature</th>
                <th>Traditional AI Agents</th>
                <th>AutoGen AI Agents</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td>Interactivity</td>
                <td>Works alone</td>
                <td>Collaborates with other agents</td>
            </tr>
            <tr>
                <td>Learning Ability</td>
                <td>Static</td>
                <td>Adaptive & iterative</td>
            </tr>
            <tr>
                <td>Workflow Handling</td>
                <td>Predefined</td>
                <td>Dynamic & evolving</td>
            </tr>
            <tr>
                <td>Human Intervention</td>
                <td>Limited</td>
                <td>Supports human-AI collaboration</td>
            </tr>
        </tbody>
</table>


The notebook uses `python-dotenv` to load OpenAI credentials and passes them into AG2 with `LLMConfig`. Code execution is not required for this healthcare consultation workflow, so no generated code is executed.


In [4]:
# Code execution is not required for this workflow.
code_execution_config = False
print("Generated code execution is disabled for this healthcare chatbot.")


Generated code execution is disabled for this healthcare chatbot.


# How AutoMed Works: Multi-Agent AI in Action
When a user interacts with AutoMed, the system does not simply generate a response—it triggers a team of AI agents, each with a specific role in processing the query. These agents collaborate in real-time, cross-validating and optimizing their responses to ensure accuracy and reliability.

For example, consider a patient experiencing persistent headaches and fatigue. Instead of offering generic advice, AutoMed intelligently activates multiple specialized AI agents:

1. **Patient Agent** – Collects user symptoms, medical history, and any ongoing treatments.
2. **Symptom Analyzer Agent** – Evaluates potential conditions such as migraines, dehydration, or anemia based on AI-driven medical knowledge.
3. **Pharmacy Agent** – Suggests remedies, over-the-counter medications, and when to seek professional medical attention.
4.  **Consultation Advisor Agent** (Decides if a doctor visit is needed) – Fetches real-time updates from trusted healthcare medical research papers.

These agents work seamlessly together, analyzing, validating, and optimizing their recommendations to deliver the most relevant, personalized, and up-to-date medical guidance. The collaborative approach ensures that users receive a well-rounded medical consultation experience, similar to interacting with multiple healthcare professionals at once, but through an AI-driven, automated system.


### Why use an OpenAI model?

The agents use the OpenAI model configured by `OPENAI_MODEL` because the workflow requires natural-language reasoning, role-specific responses, and concise summarization. The default in this notebook is `gpt-5-nano`, but you can choose another OpenAI model by setting `OPENAI_MODEL` in your `.env` file.


In [5]:
# Shared OpenAI LLM configuration was created in the setup cell.
llm_config


LLMConfig(config_list=[{'api_type': 'openai', 'model': 'gpt-5-nano', 'api_key': '**********', 'tags': [], 'stream': False}])

Now, let's understand the parameters:

- `llm_config=llm_config`: gives each AI agent access to the OpenAI model and API key loaded from the environment.
- `human_input_mode="NEVER"`: keeps this lab reproducible and non-blocking.
- `code_execution_config=False`: disables generated code execution because this workflow only needs conversation and routing.
- `ContextVariables`: stores shared workflow state that agents and tools can use across the group chat.


## What is  ConversableAgent?
- Represents an AI agent that can engage in conversations.
- Each agent has a specific role, defined by its system message.
- Uses LLM (Language Model) configurations to process responses.

 Here, each agent is assigned a specific role, allowing structured communication between AI agents and the user.
- The patient_agent represents the user and is responsible for describing symptoms and requesting medical assistance, but it does not process responses.
- The diagnosis_agent analyzes the symptoms provided by the patient and generates a concise diagnosis in a single response, ensuring clarity and brevity.
- The pharmacy_agent follows up on the diagnosis by recommending medications, but it is restricted to responding only once to prevent unnecessary repetition.
- The consultation_agent plays a critical role in determining whether the patient needs to visit a doctor, providing a final summary of the consultation along with clear next steps. To ensure structured conversation flow, the consultation_agent includes a termination condition by adding "CONSULTATION_COMPLETE" to its response, signaling the end of the consultation session. All agents are configured with the llm_config, which specifies the underlying language model (GPT-4) for processing responses. This structured setup allows for an efficient and logical multi-agent conversation, ensuring that the patient receives a diagnosis, medication recommendations, and a final decision on whether further medical consultation is necessary.


In [6]:
# Step 1: Create AI agents with defined roles.
# This example is educational and does not provide medical diagnosis or treatment.
patient_agent = ConversableAgent(
    name="patient",
    system_message="You represent the user request and start the consultation.",
    llm_config=False,
    human_input_mode="NEVER",
)

diagnosis_agent = ConversableAgent(
    name="diagnosis_agent",
    system_message=(
        "You are a cautious healthcare information assistant. Summarize symptoms, "
        "list possible non-diagnostic considerations, identify red flags, and advise seeing a clinician. "
        "Do not claim to diagnose or prescribe."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)

pharmacy_agent = ConversableAgent(
    name="pharmacy_agent",
    system_message=(
        "You provide general medication safety education. Mention interactions, allergies, "
        "contraindications, and the need to consult a licensed professional. Do not prescribe."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)

consultation_agent = ConversableAgent(
    name="consultation_agent",
    system_message=(
        "You create the final patient-friendly summary with next steps, urgent-care warnings, "
        "and a reminder that this is not medical advice."
    ),
    llm_config=llm_config,
    human_input_mode="NEVER",
)


## What is GroupChat?

- Manages a structured conversation between multiple AI agents
- Ensures turn-based speaking using speaker selection methods (e.g., round_robin)
- Prevents infinite loops by using max_round

The GroupChat class structures the interaction between multiple AI agents, ensuring a logical conversation flow. Below is a breakdown of each parameter used in the code:

- agents=[diagnosis_agent, pharmacy_agent, consultation_agent]
    - Specifies the AI agents participating in the conversation.
    - The diagnosis agent analyzes symptoms, the pharmacy agent recommends medications, and the consultation agent determines if a doctor's visit is necessary.
    - The patient agent only initiates the conversation and does not actively participate in the group chat.
- messages=[]
    - Initializes the conversation with an empty list of messages.
    - Ensures that no previous data is retained, making each consultation independent.
- max_round=5
    - Limits the conversation to five full cycles through all agents.
    - Prevents infinite loops by restricting the number of exchanges.
    - Ensures the conversation remains efficient and focused.
- speaker_selection_method="round_robin"
    - Controls the order in which agents respond.
    - Uses a "round-robin" approach, meaning each agent takes turns speaking in a structured sequence.
    - Prevents repetition or chaotic interactions, ensuring each agent contributes in a logical order.


In [7]:
# Step 2: Create a pattern-based group workflow.
# DefaultPattern uses explicit handoffs, which is a good fit for regulated domains.
context = ContextVariables(
    data={
        "domain": "healthcare_education",
        "requires_disclaimer": True,
        "urgent_care_warning": True,
    }
)

# After diagnosis_agent finishes its turn, hand the conversation to pharmacy_agent, ...
diagnosis_agent.handoffs.set_after_work(AgentTarget(pharmacy_agent))
pharmacy_agent.handoffs.set_after_work(AgentTarget(consultation_agent))
consultation_agent.handoffs.set_after_work(TerminateTarget())

healthcare_pattern = DefaultPattern(
    initial_agent=diagnosis_agent,
    agents=[diagnosis_agent, pharmacy_agent, consultation_agent],
    user_agent=patient_agent,
    context_variables=context,
    group_manager_args={"llm_config": llm_config},
)


Group chat orchestration:

- `DefaultPattern` runs a controlled workflow with explicit handoffs.
- `AgentTarget` sends the conversation to the next specialist.
- `TerminateTarget` ends the workflow after the final consultation summary.
- `ContextVariables` stores shared healthcare safety flags for the group workflow.


In [8]:
# Step 3: Run the pattern-based group chat.
manager = healthcare_pattern
print("Healthcare group chat pattern is ready.")


Healthcare group chat pattern is ready.


- `run_group_chat()` starts the multi-agent workflow.
- `DefaultPattern` coordinates the agent sequence through explicit handoffs.
- The final response should include education, next steps, urgent-care warnings, and a medical disclaimer.


In [9]:
# Step 4: Start the consultation with a sample patient message.
# Replace SAMPLE_SYMPTOMS with your own educational test case if needed.
SAMPLE_SYMPTOMS = "I have had a sore throat, mild fever, and fatigue for two days."

result = run_group_chat(
    pattern=healthcare_pattern,
    messages=(
        f"Patient symptoms: {SAMPLE_SYMPTOMS}\n"
        "Provide general health education only, include red flags, and do not diagnose."
    ),
    max_rounds=6,
)

result.process()
print(result.summary)


patient (to chat_manager):

Patient symptoms: I have had a sore throat, mild fever, and fatigue for two days.
Provide general health education only, include red flags, and do not diagnose.

--------------------------------------------------------------------------------

Next speaker: diagnosis_agent

diagnosis_agent (to chat_manager):

Thanks for sharing your symptoms. Here is general health information to help you understand what’s happening and what to do next. This is not a diagnosis and isn’t a substitute for professional care.

What your symptoms might reflect (non-diagnostic)
- Many sore throats with mild fever and fatigue are caused by viral infections (e.g., common cold, flu, or COVID-19). 
- Bacterial throat infections (such as strep throat) are also possible, but a clinician would need to evaluate symptoms and may perform tests to determine this.
- Other causes (less common) include allergies, irritants, or reflux, though these usually have other distinguishing features.

Se

## Exercise: Create a Mental Health Chatbot Using the AutoGen Library

<table border="1" cellspacing="0" align="left">
    <thead>
        <tr>
            <th>Agent</th>
            <th>Role</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td><strong>Patient Agent</strong></td>
            <td>Captures user input (mood, stress level, emotional concerns).</td>
        </tr>
        <tr>
            <td><strong>Emotion Analysis Agent</strong></td>
            <td>Identifies emotions based on user input.</td>
        </tr>
        <tr>
            <td><strong>Therapy Recommendation Agent</strong></td>
            <td>Provides relaxation techniques and coping strategies.</td>
        </tr>
    </tbody>
</table>



In [10]:
from autogen import ConversableAgent, GroupChat, GroupChatManager
from dotenv import load_dotenv

load_dotenv()

MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in your environment or .env file before running this notebook.")

# LLM Configuration
llm_config = LLMConfig(
    {
        "api_type": "openai",
        "model": MODEL_NAME,
        "api_key": OPENAI_API_KEY,
    }
)

In [11]:
# Create AI Agents with distinct roles 
patient_agent = ConversableAgent(
    name="patient",
    system_message="You describe your emotions and mental health concerns.",
    llm_config=llm_config
)

emotion_analysis_agent = ConversableAgent(
    name="emotion_analysis",
    system_message="You analyze the user's emotions based on their input."
                   "Do not provide treatment or self-care advice."
                   "Instead, just summarize the dominant emotions they may be experiencing.",
    llm_config=llm_config
)

therapy_recommendation_agent = ConversableAgent(
    name="therapy_recommendation",
    system_message="You suggest relaxation techniques and self-care methods"
                   "only based on the analysis from the Emotion Analysis Agent."
                   "Do not analyze emotions—just give recommendations based on the prior response.",
    llm_config=llm_config
)

# Create GroupChat for AI Agents 
groupchat = GroupChat(
    agents=[emotion_analysis_agent, therapy_recommendation_agent],
    messages=[], 
    max_round=3,  # Ensures the conversation does not stop too early 
    speaker_selection_method="round_robin"
)

# Create GroupChatManager 
manager = GroupChatManager(name="manager", groupchat=groupchat)

# Function to start the chatbot interaction 
def start_mental_health_chat():
    """Runs a chatbot for mental health support with distinct agent roles.""" 
    print("\nWelcome to the AI Mental Health Chatbot!") 
    user_feelings = input("How are you feeling today?")

    # Initiate conversation
    print("\nAnalyzing emotions...")
    response = patient_agent.initiate_chat(
        manager, 
        message=f"I have been feeling {user_feelings}. Can you help?"
    )

    # Ensure the therapy agent gets triggered
    if not response:  # If the initial response is empty, retry with explicit therapy agent prompt
        response = therapy_recommendation_agent.initiate_chat(
            manager, 
            message="Based on the user's emotions, please provide therapy recommendations."
        )

# Run the chatbot 
start_mental_health_chat()



Welcome to the AI Mental Health Chatbot!

Analyzing emotions...
patient (to manager):

I have been feeling happy. Can you help?

--------------------------------------------------------------------------------

Next speaker: emotion_analysis


>>>>>>>> USING AUTO REPLY...
emotion_analysis (to manager):

The user seems to be experiencing happiness.

--------------------------------------------------------------------------------

Next speaker: therapy_recommendation


>>>>>>>> USING AUTO REPLY...
therapy_recommendation (to manager):

That's wonderful to hear! Here are some self-care practices to enhance and maintain your happiness:

1. **Celebrate the Moments**: Take a moment to acknowledge and celebrate what is bringing you joy. It could be as simple as writing down three things you're grateful for.

2. **Mindful Presence**: Engage in mindfulness meditation to savor the feeling of happiness. Spend a few moments each day focused on the present, taking note of all the positive around yo

<details>
    <summary>Click here for a sample solution</summary>

```python
from autogen import ConversableAgent, GroupChat, GroupChatManager

# LLM Configuration (Replace None with actual API key if needed) 
llm_config = {"config_list": [{"model": "gpt-4o", "api_key": None}]}  # Provide OpenAI API key if required

# Create AI Agents with distinct roles 
patient_agent = ConversableAgent(
    name="patient",
    system_message="You describe your emotions and mental health concerns.",
    llm_config=llm_config
)

emotion_analysis_agent = ConversableAgent(
    name="emotion_analysis",
    system_message="You analyze the user's emotions based on their input."
                   "Do not provide treatment or self-care advice."
                   "Instead, just summarize the dominant emotions they may be experiencing.",
    llm_config=llm_config
)

therapy_recommendation_agent = ConversableAgent(
    name="therapy_recommendation",
    system_message="You suggest relaxation techniques and self-care methods"
                   "only based on the analysis from the Emotion Analysis Agent."
                   "Do not analyze emotions—just give recommendations based on the prior response.",
    llm_config=llm_config
)

# Create GroupChat for AI Agents 
groupchat = GroupChat(
    agents=[emotion_analysis_agent, therapy_recommendation_agent],
    messages=[], 
    max_round=3,  # Ensures the conversation does not stop too early 
    speaker_selection_method="round_robin"
)

# Create GroupChatManager 
manager = GroupChatManager(name="manager", groupchat=groupchat)

# Function to start the chatbot interaction 
def start_mental_health_chat():
    """Runs a chatbot for mental health support with distinct agent roles.""" 
    print("\nWelcome to the AI Mental Health Chatbot!") 
    user_feelings = input("How are you feeling today?")

    # Initiate conversation
    print("\nAnalyzing emotions...")
    response = patient_agent.initiate_chat(
        manager, 
        message=f"I have been feeling {user_feelings}. Can you help?"
    )

    # Ensure the therapy agent gets triggered
    if not response:  # If the initial response is empty, retry with explicit therapy agent prompt
        response = therapy_recommendation_agent.initiate_chat(
            manager, 
            message="Based on the user's emotions, please provide therapy recommendations."
        )

# Run the chatbot 
start_mental_health_chat()


## Authors


[Jigisha Barbhaya](https://www.linkedin.com/in/jigisha-barbhaya/) has always been driven by a passion for sharing knowledge and helping others learn about data science. This belief—that everyone should have the opportunity to learn about the field, regardless of their background or experience—has inspired her work as a learning content provider. Educational materials that are both accessible and engaging have been created and shared to make learning about data science easier for everyone.


### Other Contributors


[Faranak Heidari](https://author.skills.network/instructors/faranak_heidari) is a Data Scientist at IBM.


## Change Log

<details>
    <summary>Click here for the changelog</summary>

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2025-07-22|0.1|Jigisha Barbhaya|Initial version created|
|2025-07-22|0.2|Steve Ryan|ID review|
|2025-07-22|0.3|Andrea Hansis|Content QA review|

</details>

---



Copyright © IBM Corporation. All rights reserved.
